In [1]:
import h5py

file_path = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\raw\insat\Apr26_176936\3RIMG_01FEB2023_0545_L2G_AOD_V02R00.h5"

with h5py.File(file_path, "r") as f:
    # List top-level groups/datasets
    print("Keys:")
    print(list(f.keys()))

Keys:
['AOD', 'latitude', 'longitude', 'time']


In [5]:
def explore(name, obj):
    print(name)
    if isinstance(obj, h5py.Dataset):
        print("  Shape:", obj.shape)
        print("  Dtype:", obj.dtype)
        print()

with h5py.File(file_path, "r") as f:
    f.visititems(explore)

AOD
  Shape: (1, 551, 551)
  Dtype: float32

latitude
  Shape: (551,)
  Dtype: float64

longitude
  Shape: (551,)
  Dtype: float64

time
  Shape: (1,)
  Dtype: float64



In [4]:
with h5py.File(file_path, "r") as f:
    data = f["time"][:]
    print(data[:5])   # first values

[12142425.]


In [6]:
import h5py
import pandas as pd
import numpy as np



with h5py.File(file_path, "r") as f:
    aod = f["AOD"][:]          # (1, 551, 551)
    lat = f["latitude"][:]     # (551,)
    lon = f["longitude"][:]    # (551,)
    time = f["time"][:]        # (1,)

# Take small slice (head)
aod_head = aod[0, :5, :5]   # first 5 lat, first 5 lon

# Create grid for that slice
lat_head = lat[:5]
lon_head = lon[:5]

lat_grid, lon_grid = np.meshgrid(lat_head, lon_head, indexing="ij")

# Make dataframe
head_df = pd.DataFrame({
    "time": time[0],
    "latitude": lat_grid.flatten(),
    "longitude": lon_grid.flatten(),
    "AOD": aod_head.flatten()
})

print(head_df)

          time  latitude  longitude    AOD
0   12142425.0     45.05      45.05 -999.0
1   12142425.0     45.05      45.15 -999.0
2   12142425.0     45.05      45.25 -999.0
3   12142425.0     45.05      45.35 -999.0
4   12142425.0     45.05      45.45 -999.0
5   12142425.0     44.95      45.05 -999.0
6   12142425.0     44.95      45.15 -999.0
7   12142425.0     44.95      45.25 -999.0
8   12142425.0     44.95      45.35 -999.0
9   12142425.0     44.95      45.45 -999.0
10  12142425.0     44.85      45.05 -999.0
11  12142425.0     44.85      45.15 -999.0
12  12142425.0     44.85      45.25 -999.0
13  12142425.0     44.85      45.35 -999.0
14  12142425.0     44.85      45.45 -999.0
15  12142425.0     44.75      45.05 -999.0
16  12142425.0     44.75      45.15 -999.0
17  12142425.0     44.75      45.25 -999.0
18  12142425.0     44.75      45.35 -999.0
19  12142425.0     44.75      45.45 -999.0
20  12142425.0     44.65      45.05 -999.0
21  12142425.0     44.65      45.15 -999.0
22  1214242

In [1]:
import h5py
import numpy as np
import pandas as pd
import os
from glob import glob
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime

# ==============================
# SETTINGS
# ==============================
base_folder = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\raw\insat"
output_file = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\insat_aod_merged.parquet"

LAT_MIN, LAT_MAX = 6, 38
LON_MIN, LON_MAX = 68, 98

# ==============================
# GET ALL H5 FILES
# ==============================
h5_files = glob(os.path.join(base_folder, "**", "*.h5"), recursive=True)
print("Total H5 files:", len(h5_files))

writer = None

# ==============================
# FUNCTION: Extract datetime from filename
# ==============================
def get_datetime_from_filename(file_path):
    filename = os.path.basename(file_path)
    
    # Example: 3RIMG_01FEB2023_0545_L2G_AOD_V02R00.h5
    parts = filename.split("_")
    
    date_str = parts[1]   # 01FEB2023
    time_str = parts[2]   # 0545
    
    dt = datetime.strptime(date_str + time_str, "%d%b%Y%H%M")
    return dt

# ==============================
# PROCESS FILES
# ==============================
for i, file in enumerate(h5_files):
    try:
        with h5py.File(file, "r") as f:
            aod = f["AOD"][:]          
            lat = f["latitude"][:]     
            lon = f["longitude"][:]     

            aod = aod[0]

            # Create grid
            lat_grid, lon_grid = np.meshgrid(lat, lon, indexing="ij")

            # Flatten
            df = pd.DataFrame({
                "latitude": lat_grid.flatten(),
                "longitude": lon_grid.flatten(),
                "AOD": aod.flatten()
            })

            # Filter India
            df = df[
                (df["latitude"] >= LAT_MIN) & (df["latitude"] <= LAT_MAX) &
                (df["longitude"] >= LON_MIN) & (df["longitude"] <= LON_MAX)
            ]

            # Replace missing values
            df["AOD"] = df["AOD"].replace(-999, np.nan)

            # Get datetime from filename
            file_datetime = get_datetime_from_filename(file)
            df["datetime"] = file_datetime

            # Final columns
            df = df[["datetime", "latitude", "longitude", "AOD"]]

            # Convert to parquet table
            table = pa.Table.from_pandas(df)

            # Write incrementally
            if writer is None:
                writer = pq.ParquetWriter(output_file, table.schema)
            
            writer.write_table(table)

            print(f"Processed ({i+1}/{len(h5_files)}):", os.path.basename(file))

    except Exception as e:
        print("Error in:", file)
        print(e)

# Close writer
if writer:
    writer.close()

print("\nFinal file saved as:", output_file)

Total H5 files: 4745
Processed (1/4745): 3RIMG_01FEB2023_0545_L2G_AOD_V02R00.h5
Processed (2/4745): 3RIMG_01FEB2023_0615_L2G_AOD_V02R00.h5
Processed (3/4745): 3RIMG_01FEB2023_0645_L2G_AOD_V02R00.h5
Processed (4/4745): 3RIMG_01FEB2023_0745_L2G_AOD_V02R00.h5
Processed (5/4745): 3RIMG_01FEB2023_0815_L2G_AOD_V02R00.h5
Processed (6/4745): 3RIMG_01FEB2023_0845_L2G_AOD_V02R00.h5
Processed (7/4745): 3RIMG_01JAN2023_0545_L2G_AOD_V02R00.h5
Processed (8/4745): 3RIMG_01JAN2023_0615_L2G_AOD_V02R00.h5
Processed (9/4745): 3RIMG_01JAN2023_0645_L2G_AOD_V02R00.h5
Processed (10/4745): 3RIMG_01JAN2023_0715_L2G_AOD_V02R00.h5
Processed (11/4745): 3RIMG_01JAN2023_0745_L2G_AOD_V02R00.h5
Processed (12/4745): 3RIMG_01JAN2023_0815_L2G_AOD_V02R00.h5
Processed (13/4745): 3RIMG_01JAN2023_0845_L2G_AOD_V02R00.h5
Processed (14/4745): 3RIMG_01MAR2023_0545_L2G_AOD_V02R00.h5
Processed (15/4745): 3RIMG_01MAR2023_0615_L2G_AOD_V02R00.h5
Processed (16/4745): 3RIMG_01MAR2023_0645_L2G_AOD_V02R00.h5
Processed (17/4745): 3RIMG_0

In [2]:
import pyarrow.parquet as pq

file_path = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\insat_aod_merged.parquet"

parquet_file = pq.ParquetFile(file_path)

print("Row groups:", parquet_file.num_row_groups)

# Read ONLY first row group (safe)
table = parquet_file.read_row_group(0)

df_sample = table.to_pandas()

print(df_sample.head())

Row groups: 4745
                 datetime  latitude  longitude  AOD
39351 2023-02-01 05:45:00     37.95      68.05  NaN
39352 2023-02-01 05:45:00     37.95      68.15  NaN
39353 2023-02-01 05:45:00     37.95      68.25  NaN
39354 2023-02-01 05:45:00     37.95      68.35  NaN
39355 2023-02-01 05:45:00     37.95      68.45  NaN


In [3]:
import pyarrow.parquet as pq

file = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\insat_aod_merged.parquet"
parquet_file = pq.ParquetFile(file)

print("Number of row groups:", parquet_file.num_row_groups)
print("Schema:")
print(parquet_file.schema)

Number of row groups: 4745
Schema:
required group field_id=-1 schema {
  optional int64 field_id=-1 datetime (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional double field_id=-1 latitude;
  optional double field_id=-1 longitude;
  optional float field_id=-1 AOD;
  optional int64 field_id=-1 __index_level_0__;
}



In [4]:
import os

# File size
size_mb = os.path.getsize(file) / (1024 * 1024)
print("File size (MB):", round(size_mb, 2))

# Total rows
total_rows = parquet_file.metadata.num_rows
print("Total rows:", total_rows)

File size (MB): 3293.39
Total rows: 455520000


In [5]:
import pandas as pd

unique_dates = set()

for i in range(parquet_file.num_row_groups):
    df = parquet_file.read_row_group(i, columns=["datetime"]).to_pandas()
    unique_dates.update(df["datetime"].dt.date.unique())

print("Unique Dates:")
print(sorted(unique_dates))
print("Total Unique Dates:", len(unique_dates))

Unique Dates:
[datetime.date(2023, 1, 1), datetime.date(2023, 1, 2), datetime.date(2023, 1, 3), datetime.date(2023, 1, 4), datetime.date(2023, 1, 5), datetime.date(2023, 1, 6), datetime.date(2023, 1, 7), datetime.date(2023, 1, 8), datetime.date(2023, 1, 9), datetime.date(2023, 1, 10), datetime.date(2023, 1, 11), datetime.date(2023, 1, 12), datetime.date(2023, 1, 13), datetime.date(2023, 1, 14), datetime.date(2023, 1, 15), datetime.date(2023, 1, 16), datetime.date(2023, 1, 17), datetime.date(2023, 1, 18), datetime.date(2023, 1, 19), datetime.date(2023, 1, 20), datetime.date(2023, 1, 21), datetime.date(2023, 1, 22), datetime.date(2023, 1, 23), datetime.date(2023, 1, 24), datetime.date(2023, 1, 25), datetime.date(2023, 1, 26), datetime.date(2023, 1, 27), datetime.date(2023, 1, 28), datetime.date(2023, 1, 29), datetime.date(2023, 1, 30), datetime.date(2023, 1, 31), datetime.date(2023, 2, 1), datetime.date(2023, 2, 2), datetime.date(2023, 2, 3), datetime.date(2023, 2, 4), datetime.date(2023

In [6]:
unique_times = set()

for i in range(parquet_file.num_row_groups):
    df = parquet_file.read_row_group(i, columns=["datetime"]).to_pandas()
    unique_times.update(df["datetime"].dt.time.unique())

print("Unique Times:")
print(sorted(unique_times))
print("Total Unique Times:", len(unique_times))

Unique Times:
[datetime.time(5, 45), datetime.time(6, 15), datetime.time(6, 45), datetime.time(7, 15), datetime.time(7, 45), datetime.time(8, 15), datetime.time(8, 45)]
Total Unique Times: 7


In [7]:
min_lat, max_lat = 999, -999
min_lon, max_lon = 999, -999

for i in range(parquet_file.num_row_groups):
    df = parquet_file.read_row_group(i, columns=["latitude", "longitude"]).to_pandas()
    
    min_lat = min(min_lat, df["latitude"].min())
    max_lat = max(max_lat, df["latitude"].max())
    min_lon = min(min_lon, df["longitude"].min())
    max_lon = max(max_lon, df["longitude"].max())

print("Latitude Min:", min_lat)
print("Latitude Max:", max_lat)
print("Longitude Min:", min_lon)
print("Longitude Max:", max_lon)

Latitude Min: 6.049999999999997
Latitude Max: 37.949999999999996
Longitude Min: 68.05
Longitude Max: 97.95


In [8]:
null_count = 0
total_count = 0

for i in range(parquet_file.num_row_groups):
    df = parquet_file.read_row_group(i, columns=["AOD"]).to_pandas()
    
    null_count += df["AOD"].isna().sum()
    total_count += len(df)

print("Total rows:", total_count)
print("Null AOD rows:", null_count)
print("Null %:", (null_count / total_count) * 100)

Total rows: 455520000
Null AOD rows: 335204568
Null %: 73.58723393045311


In [9]:
rows_per_time = {}

for i in range(parquet_file.num_row_groups):
    df = parquet_file.read_row_group(i, columns=["datetime"]).to_pandas()
    counts = df["datetime"].value_counts()
    
    for k, v in counts.items():
        rows_per_time[k] = v

print("Sample rows per timestamp:")
print(list(rows_per_time.items())[:5])

Sample rows per timestamp:
[(Timestamp('2023-02-01 05:45:00'), 96000), (Timestamp('2023-02-01 06:15:00'), 96000), (Timestamp('2023-02-01 06:45:00'), 96000), (Timestamp('2023-02-01 07:45:00'), 96000), (Timestamp('2023-02-01 08:15:00'), 96000)]


In [11]:
import pandas as pd

file_path = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\india_clipped\insat_india.parquet"

insat_df = pd.read_parquet(file_path)

# Rows and Columns
print("Rows, Columns:", insat_df.shape)

insat_df['datetime'] = pd.to_datetime(insat_df['datetime'])
insat_df['hour'] = insat_df['datetime'].dt.hour
insat_df['month'] = insat_df['datetime'].dt.month

missing_by_hour = insat_df.groupby('hour')['AOD'].apply(
    lambda x: x.isna().mean() * 100
)

print(missing_by_hour)

Rows, Columns: (160812795, 4)
hour
5    64.345413
6    69.358687
7    68.814755
8    63.528856
Name: AOD, dtype: float64


In [12]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
import os

input_file = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\india_clipped\insat_india.parquet"
output_file = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\india_clipped\insat_india_clean.parquet"

os.makedirs(os.path.dirname(output_file), exist_ok=True)

pf = pq.ParquetFile(input_file)
writer = None
total_rows = 0
kept_rows = 0

for i in range(pf.num_row_groups):
    print(f"Processing row group {i+1}/{pf.num_row_groups}")
    
    df = pf.read_row_group(i).to_pandas()
    
    # Convert datetime
    df["datetime"] = pd.to_datetime(df["datetime"])
    
    # Extract hour
    df["hour"] = df["datetime"].dt.hour
    
    # Rule 1 — Keep daytime AOD only
    df = df[df["hour"].between(5, 14)]
    
    # Rule 2 — Drop NaN AOD
    df = df.dropna(subset=["AOD"])
    
    # Rule 3 — Physical AOD range
    df = df[(df["AOD"] >= 0.1) & (df["AOD"] <= 3.0)]
    
    df = df.drop(columns=["hour"])
    
    total_rows += pf.metadata.row_group(i).num_rows
    kept_rows += len(df)
    
    if len(df) == 0:
        continue

    table = pa.Table.from_pandas(df)

    if writer is None:
        writer = pq.ParquetWriter(output_file, table.schema)

    writer.write_table(table)

if writer:
    writer.close()

print("\nOriginal rows:", total_rows)
print("Retained rows:", kept_rows)
print("Retention %:", (kept_rows / total_rows) * 100)
print("Saved to:", output_file)

Processing row group 1/4745
Processing row group 2/4745
Processing row group 3/4745
Processing row group 4/4745
Processing row group 5/4745
Processing row group 6/4745
Processing row group 7/4745
Processing row group 8/4745
Processing row group 9/4745
Processing row group 10/4745
Processing row group 11/4745
Processing row group 12/4745
Processing row group 13/4745
Processing row group 14/4745
Processing row group 15/4745
Processing row group 16/4745
Processing row group 17/4745
Processing row group 18/4745
Processing row group 19/4745
Processing row group 20/4745
Processing row group 21/4745
Processing row group 22/4745
Processing row group 23/4745
Processing row group 24/4745
Processing row group 25/4745
Processing row group 26/4745
Processing row group 27/4745
Processing row group 28/4745
Processing row group 29/4745
Processing row group 30/4745
Processing row group 31/4745
Processing row group 32/4745
Processing row group 33/4745
Processing row group 34/4745
Processing row group 35

In [13]:
import pandas as pd

file_path = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\india_clipped\insat_india_clean.parquet"

merra2_df = pd.read_parquet(file_path)

# Rows and Columns
print("Rows, Columns:", merra2_df.shape)

Rows, Columns: (51360120, 4)


In [15]:
import pyarrow.parquet as pq

file_path = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\india_clipped\insat_india_clean.parquet"
parquet_file = pq.ParquetFile(file_path)


print("Number of row groups:", parquet_file.num_row_groups)
print("Schema:")
print(parquet_file.schema)

Number of row groups: 4745
Schema:
required group field_id=-1 schema {
  optional int64 field_id=-1 datetime (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional double field_id=-1 latitude;
  optional double field_id=-1 longitude;
  optional float field_id=-1 AOD;
  optional int64 field_id=-1 __index_level_0__;
}



In [16]:
import pandas as pd

unique_dates = set()

for i in range(parquet_file.num_row_groups):
    df = parquet_file.read_row_group(i, columns=["datetime"]).to_pandas()
    unique_dates.update(df["datetime"].dt.date.unique())

print("Unique Dates:")
print(sorted(unique_dates))
print("Total Unique Dates:", len(unique_dates))

Unique Dates:
[datetime.date(2023, 1, 1), datetime.date(2023, 1, 2), datetime.date(2023, 1, 3), datetime.date(2023, 1, 4), datetime.date(2023, 1, 5), datetime.date(2023, 1, 6), datetime.date(2023, 1, 7), datetime.date(2023, 1, 8), datetime.date(2023, 1, 9), datetime.date(2023, 1, 10), datetime.date(2023, 1, 11), datetime.date(2023, 1, 12), datetime.date(2023, 1, 13), datetime.date(2023, 1, 14), datetime.date(2023, 1, 15), datetime.date(2023, 1, 16), datetime.date(2023, 1, 17), datetime.date(2023, 1, 18), datetime.date(2023, 1, 19), datetime.date(2023, 1, 20), datetime.date(2023, 1, 21), datetime.date(2023, 1, 22), datetime.date(2023, 1, 23), datetime.date(2023, 1, 24), datetime.date(2023, 1, 25), datetime.date(2023, 1, 26), datetime.date(2023, 1, 27), datetime.date(2023, 1, 28), datetime.date(2023, 1, 29), datetime.date(2023, 1, 30), datetime.date(2023, 1, 31), datetime.date(2023, 2, 1), datetime.date(2023, 2, 2), datetime.date(2023, 2, 3), datetime.date(2023, 2, 4), datetime.date(2023

In [17]:
min_lat, max_lat = 999, -999
min_lon, max_lon = 999, -999

for i in range(parquet_file.num_row_groups):
    df = parquet_file.read_row_group(i, columns=["latitude", "longitude"]).to_pandas()
    
    min_lat = min(min_lat, df["latitude"].min())
    max_lat = max(max_lat, df["latitude"].max())
    min_lon = min(min_lon, df["longitude"].min())
    max_lon = max(max_lon, df["longitude"].max())

print("Latitude Min:", min_lat)
print("Latitude Max:", max_lat)
print("Longitude Min:", min_lon)
print("Longitude Max:", max_lon)

Latitude Min: 7.549999999999997
Latitude Max: 35.949999999999996
Longitude Min: 68.05
Longitude Max: 97.85


In [18]:
null_count = 0
total_count = 0

for i in range(parquet_file.num_row_groups):
    df = parquet_file.read_row_group(i, columns=["AOD"]).to_pandas()
    
    null_count += df["AOD"].isna().sum()
    total_count += len(df)

print("Total rows:", total_count)
print("Null AOD rows:", null_count)
print("Null %:", (null_count / total_count) * 100)

Total rows: 51360120
Null AOD rows: 0
Null %: 0.0


In [5]:
import dask.dataframe as dd
df = dd.read_parquet(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\insat_daily\insatdaily_grid.parquet")

In [6]:
rows = df.shape[0].compute()

# Columns
cols = df.shape[1]

print("Shape:", (rows, cols))

Shape: (9928398, 8)


In [7]:
df.isnull().sum().compute()

lat_round       0
lon_round       0
date            0
AOD_mean        0
AOD_max         0
AOD_p75         0
AOD_count       0
AOD_coverage    0
dtype: int64

In [1]:
import dask.dataframe as dd
import os
import shutil

original_file = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\insat_daily\insat_daily_grid.parquet"
temp_file = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\insat_daily\insat_indiadaily.parquet"

# Load parquet
df = dd.read_parquet(original_file)

# Drop AOD_std if exists
if 'AOD_std' in df.columns:
    df = df.drop(columns=['AOD_std'])

print("Updated columns:", df.columns)

# Save to temporary parquet
df.to_parquet(temp_file, write_index=False)

# Delete original file
if os.path.exists(original_file):
    os.remove(original_file)

# Rename temp file to original file name
os.rename(temp_file, original_file)

print("Original file updated (AOD_std removed).")

Updated columns: Index(['lat_round', 'lon_round', 'date', 'AOD_mean', 'AOD_max', 'AOD_p75',
       'AOD_count', 'AOD_coverage'],
      dtype='object')
Original file updated (AOD_std removed).


In [8]:
df.head()

,lat_round,lon_round,date,AOD_mean,AOD_max,AOD_p75,AOD_count,AOD_coverage
0,7.55,77.35,2023-01-02,0.171448,0.208544,0.171448,2,0.285714
1,7.55,77.35,2023-01-03,0.191858,0.263068,0.191858,6,0.857143
2,7.55,77.35,2023-01-04,0.294082,0.314460,0.294082,4,0.571429
3,7.55,77.35,2023-01-05,0.233201,0.268513,0.233201,7,1.000000
4,7.55,77.35,2023-01-07,0.305637,0.387881,0.305637,7,1.000000


In [9]:
import dask.dataframe as dd

file = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\insat_daily\insatdaily_grid.parquet"

df = dd.read_parquet(file, columns=['lat_round', 'lon_round'])

# Drop duplicates
unique_grids = df.drop_duplicates()

# Count unique pairs
count = unique_grids.shape[0].compute()

print("Unique lat-lon grid points:", count)

Unique lat-lon grid points: 31033


In [23]:
import pyarrow.parquet as pq

file_path = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\insat_daily\insat_daily_grid.parquet"
parquet_file = pq.ParquetFile(file_path)


print("Number of row groups:", parquet_file.num_row_groups)
print("Schema:")
print(parquet_file.schema)

Number of row groups: 10
Schema:
required group field_id=-1 schema {
  optional double field_id=-1 lat_round;
  optional double field_id=-1 lon_round;
  optional int32 field_id=-1 date (Date);
  optional float field_id=-1 AOD_mean;
  optional float field_id=-1 AOD_max;
  optional float field_id=-1 AOD_std;
  optional float field_id=-1 AOD_p75;
  optional int64 field_id=-1 AOD_count;
  optional double field_id=-1 AOD_coverage;
}



In [29]:
import pandas as pd

df = pd.read_parquet(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\insat_daily\insat_daily_grid.parquet", columns=["date"])

unique_dates = df["date"].drop_duplicates().sort_values()

print(unique_dates)
print("Total unique dates:", len(unique_dates))

279      2023-01-01
0        2023-01-02
1        2023-01-03
2        2023-01-04
3        2023-01-05
            ...    
277      2024-12-27
5277     2024-12-28
278      2024-12-29
11825    2024-12-30
68782    2024-12-31
Name: date, Length: 706, dtype: object
Total unique dates: 706


In [28]:
min_lat, max_lat = 999, -999
min_lon, max_lon = 999, -999

for i in range(parquet_file.num_row_groups):
    df = parquet_file.read_row_group(i, columns=["lat_round", "lon_round"]).to_pandas()
    
    min_lat = min(min_lat, df["lat_round"].min())
    max_lat = max(max_lat, df["lat_round"].max())
    min_lon = min(min_lon, df["lon_round"].min())
    max_lon = max(max_lon, df["lon_round"].max())

print("Latitude Min:", min_lat)
print("Latitude Max:", max_lat)
print("Longitude Min:", min_lon)
print("Longitude Max:", max_lon)

Latitude Min: 7.55
Latitude Max: 35.15
Longitude Min: 68.05
Longitude Max: 97.85


In [22]:
import pandas as pd

FILE = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\spatial_join\insat_station_daily.parquet"

df = pd.read_parquet(FILE)

# =========================
# BASIC INFO
# =========================
print("Total rows:", len(df))
print("Columns:", df.columns.tolist())

# =========================
# UNIQUE DATES
# =========================
unique_dates = df['date'].nunique()
print("\nUnique dates:", unique_dates)

print("Date range:", df['date'].min(), "to", df['date'].max())

# =========================
# UNIQUE STATIONS (proxy for lat-lon)
# =========================
unique_stations = df['station_name'].nunique()
print("\nUnique stations:", unique_stations)

# =========================
# CHECK DUPLICATES (important)
# =========================
dup = df.duplicated(subset=['station_name', 'date']).sum()
print("\nDuplicate (station_name, date) rows:", dup)

# =========================
# MISSING VALUES
# =========================
print("\nMissing values (%):")
missing = df.isna().mean() * 100
print(missing.sort_values(ascending=False))

# =========================
# FULLY MISSING ROWS (AOD missing case)
# =========================
missing_rows = df[
    df[['AOD_mean', 'AOD_max', 'AOD_p75']].isna().all(axis=1)
]

print("\nRows with NO AOD data:", len(missing_rows))
print("Percentage:", (len(missing_rows)/len(df))*100)

# =========================
# EXPECTED VS ACTUAL
# =========================
expected = unique_stations * unique_dates
print("\nExpected rows:", expected)
print("Actual rows  :", len(df))

if expected == len(df):
    print("✅ Perfect grid (no missing station-date pairs)")
else:
    print("⚠️ Missing station-date combinations detected")

Total rows: 68482
Columns: ['station_name', 'date', 'n_pixels', 'AOD_mean', 'AOD_max', 'AOD_p75', 'AOD_count', 'AOD_coverage']

Unique dates: 706
Date range: 2023-01-01 to 2024-12-31

Unique stations: 97

Duplicate (station_name, date) rows: 0

Missing values (%):
AOD_mean        41.761339
AOD_max         41.761339
AOD_p75         41.761339
station_name     0.000000
n_pixels         0.000000
date             0.000000
AOD_count        0.000000
AOD_coverage     0.000000
dtype: float64

Rows with NO AOD data: 28599
Percentage: 41.76133874594784

Expected rows: 68482
Actual rows  : 68482
✅ Perfect grid (no missing station-date pairs)


In [23]:
df.head()

,station_name,date,n_pixels,AOD_mean,AOD_max,AOD_p75,AOD_count,AOD_coverage
0,Sector 62 Noida,2023-01-01,5,0.978464,1.244282,0.978464,7,1.000000
1,ITO Delhi,2023-01-01,6,0.918637,1.056489,0.918637,7,1.000000
2,DTU Delhi,2023-01-01,7,0.835057,0.910091,0.835057,4,0.571429
3,R K Puram Delhi,2023-01-01,5,0.813867,0.877527,0.813867,4,0.571429
4,Punjabi Bagh Delhi,2023-01-01,6,0.794897,0.870788,0.794897,4,0.571429


In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Is missingness seasonal? (it should be — monsoon will show ~80-90% missing)
df['month'] = pd.to_datetime(df['date']).dt.month
monthly_missing = df.groupby('month')['AOD_mean'].apply(
    lambda x: x.isna().mean() * 100
)
print(monthly_missing)
# Expected pattern:
# Jan-Mar (Winter)     : 20-30% missing
# Jun-Sep (Monsoon)    : 70-90% missing  ← clouds dominate
# Oct-Dec (Post-monsoon): 30-40% missing

# 2. Is missingness station-specific?
station_missing = df.groupby('station_name')['AOD_mean'].apply(
    lambda x: x.isna().mean() * 100
).sort_values(ascending=False)
print(station_missing.head(10))
# If some stations show 80%+ missing — they may be in chronically cloudy regions
# Flag these stations — their AOD signal will be weak regardless

# 3. Does PM25 differ on missing vs present AOD days?
# pm_with_aod    = df[df['AOD_mean'].notna()]['pm25'].mean()
# pm_without_aod = df[df['AOD_mean'].isna()]['pm25'].mean()
# print(f"PM25 when AOD present : {pm_with_aod:.1f} µg/m³")
# print(f"PM25 when AOD missing : {pm_without_aod:.1f} µg/m³")
# This tells you how much information you're losing on cloud days

month
1     23.511806
2     11.557244
3     17.326239
4     19.467354
5     31.691485
6     61.422680
7     92.750249
8     89.590954
9     71.907216
10    28.208319
11    26.546392
12    25.318889
Name: AOD_mean, dtype: float64
station_name
Belur Math Howrah                 100.000000
Collectorate Jodhpur               99.716714
Bombay Castel Ooty                 70.113314
Lumpyngngad Shillong Meghalaya     64.589235
Sanjay Palace Agra                 62.464589
BTM Layout Bengaluru               62.039660
Silk Board Bengaluru               60.623229
Jayanagar 5th Block Bengaluru      60.481586
Hebbal Bengaluru                   60.198300
Peenya Bengaluru                   59.065156
Name: AOD_mean, dtype: float64


In [25]:
# Calculate per-station AOD missing rate
station_missing_rate = df.groupby('station_name')['AOD_mean'].apply(
    lambda x: (x == -1).mean()  # after sentinel fill
).reset_index()
station_missing_rate.columns = ['station_name', 'aod_missing_rate']

# Categorize stations by AOD reliability
def station_aod_category(rate):
    if rate >= 0.95:
        return 'dead'          # AOD useless — 95%+ missing
    elif rate >= 0.60:
        return 'poor'          # AOD weak signal
    elif rate >= 0.40:
        return 'moderate'      # monsoon-affected but usable
    else:
        return 'good'          # reliable AOD

station_missing_rate['aod_quality_cat'] = station_missing_rate[
    'aod_missing_rate'
].apply(station_aod_category)

print(station_missing_rate['aod_quality_cat'].value_counts())

# Merge this back into main df as a feature
df = df.merge(station_missing_rate, on='station_name', how='left')

# Encode category
cat_map = {'dead': 0, 'poor': 1, 'moderate': 2, 'good': 3}
df['station_aod_quality'] = df['aod_quality_cat'].map(cat_map)

aod_quality_cat
good    97
Name: count, dtype: int64


In [26]:
dead_stations = station_missing_rate[
    station_missing_rate['aod_quality_cat'] == 'dead'
]['station_name'].tolist()

poor_stations = station_missing_rate[
    station_missing_rate['aod_quality_cat'] == 'poor'
]['station_name'].tolist()

print(f"Dead stations ({len(dead_stations)}):", dead_stations)
print(f"Poor stations ({len(poor_stations)}):", poor_stations)



Dead stations (0): []
Poor stations (0): []


In [27]:
# Compute mean AOD per station per month — only from valid (non -1) days
valid_aod = df[df['AOD_mean'] != -1].copy()

station_month_aod = valid_aod.groupby(
    ['station_name', 'month']
)['AOD_mean'].agg(['mean', 'count']).reset_index()

station_month_aod.columns = [
    'station_name', 'month', 
    'station_month_aod_clim',   # climatological AOD for this station-month
    'station_month_aod_n'       # how many days this is based on
]

df = df.merge(station_month_aod, on=['station_name', 'month'], how='left')

# If station-month climatology itself is missing (dead station + month)
# fill with overall India monthly mean
india_month_aod = valid_aod.groupby('month')['AOD_mean'].mean()
df['station_month_aod_clim'] = df.apply(
    lambda row: india_month_aod.get(row['month'], 0.3)
    if pd.isna(row['station_month_aod_clim'])
    else row['station_month_aod_clim'],
    axis=1
)

# Reliability weight — how trustworthy is that climatology?
# Low n = unreliable climatology
df['aod_clim_reliability'] = np.minimum(
    df['station_month_aod_n'].fillna(0) / 30, 1.0
)  # caps at 1.0 after 30 observations

In [28]:
blr_stations = [s for s in df['station_name'].unique() if 'Bengaluru' in s]

blr_missing = df[df['station_name'].isin(blr_stations)].groupby(
    'month'
)['AOD_mean'].apply(lambda x: (x == -1).mean() * 100)

print("Bengaluru AOD missing by month:")
print(blr_missing.round(1))

# If non-monsoon months (Jan-May, Oct-Dec) still show >40% missing
# it may be a retrieval geometry issue — flag these stations differently
blr_nonmonsoon_missing = blr_missing[[1,2,3,4,5,10,11,12]].mean()
print(f"\nBengaluru non-monsoon missing: {blr_nonmonsoon_missing:.1f}%")

if blr_nonmonsoon_missing > 40:
    print("⚠ Geometric/systematic retrieval issue — treat like poor stations")
    df.loc[df['station_name'].isin(blr_stations), 'station_aod_quality'] = 1

Bengaluru AOD missing by month:
month
1     0.0
2     0.0
3     0.0
4     0.0
5     0.0
6     0.0
7     0.0
8     0.0
9     0.0
10    0.0
11    0.0
12    0.0
Name: AOD_mean, dtype: float64

Bengaluru non-monsoon missing: 0.0%


In [29]:
# More nuanced than a single AOD_missing flag
# Model needs to know: "missing in July" is very different from "missing in January"

df['aod_missing_winter']   = ((df['AOD_mean'] == -1) & 
                               df['month'].isin([12,1,2])).astype(int)
df['aod_missing_premonsoon']= ((df['AOD_mean'] == -1) & 
                                df['month'].isin([3,4,5])).astype(int)
df['aod_missing_monsoon']   = ((df['AOD_mean'] == -1) & 
                                df['month'].isin([6,7,8,9])).astype(int)
df['aod_missing_postmonsoon']= ((df['AOD_mean'] == -1) & 
                                 df['month'].isin([10,11])).astype(int)

In [31]:
df.head()

,station_name,date,n_pixels,AOD_mean,AOD_max,AOD_p75,AOD_count,AOD_coverage,month,aod_missing_rate,aod_quality_cat,station_aod_quality,station_month_aod_clim,station_month_aod_n,aod_clim_reliability,aod_missing_winter,aod_missing_premonsoon,aod_missing_monsoon,aod_missing_postmonsoon
0,Sector 62 Noida,2023-01-01,5,0.978464,1.244282,0.978464,7,1.000000,1,0.0,good,3,1.151048,47,1.0,0,0,0,0
1,ITO Delhi,2023-01-01,6,0.918637,1.056489,0.918637,7,1.000000,1,0.0,good,3,1.096681,48,1.0,0,0,0,0
2,DTU Delhi,2023-01-01,7,0.835057,0.910091,0.835057,4,0.571429,1,0.0,good,3,0.938084,43,1.0,0,0,0,0
3,R K Puram Delhi,2023-01-01,5,0.813867,0.877527,0.813867,4,0.571429,1,0.0,good,3,0.994317,42,1.0,0,0,0,0
4,Punjabi Bagh Delhi,2023-01-01,6,0.794897,0.870788,0.794897,4,0.571429,1,0.0,good,3,0.933955,41,1.0,0,0,0,0


In [21]:
df.isnull().sum()

station_name                   0
date                           0
n_pixels                       0
AOD_mean                   28599
AOD_max                    28599
AOD_p75                    28599
AOD_count                      0
AOD_coverage                   0
month                          0
aod_missing_rate               0
aod_quality_cat                0
station_aod_quality            0
station_month_aod_clim         0
station_month_aod_n            0
aod_clim_reliability           0
aod_missing_winter             0
aod_missing_premonsoon         0
aod_missing_monsoon            0
aod_missing_postmonsoon        0
dtype: int64

In [32]:
df.shape

(68482, 19)

In [2]:
import dask.dataframe as dd
df = dd.read_parquet(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\spatial_join\insat_station_daily.parquet")
df1 = dd.read_parquet(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\spatial_join\merra_station_daily.parquet")

In [4]:
df.head()

,station_name,date,n_pixels,AOD_mean,AOD_max,AOD_p75,AOD_count,AOD_coverage
0,Sector 62 Noida,2023-01-01,5,0.978464,1.244282,0.978464,7,1.000000
1,ITO Delhi,2023-01-01,6,0.918637,1.056489,0.918637,7,1.000000
2,DTU Delhi,2023-01-01,7,0.835057,0.910091,0.835057,4,0.571429
3,R K Puram Delhi,2023-01-01,5,0.813867,0.877527,0.813867,4,0.571429
4,Punjabi Bagh Delhi,2023-01-01,6,0.794897,0.870788,0.794897,4,0.571429


In [27]:
print(df.columns)
print(df.shape[0].compute())

Index(['station_name', 'date', 'n_pixels', 'AOD_mean', 'AOD_max', 'AOD_p75',
       'AOD_count', 'AOD_coverage'],
      dtype='object')
68482


In [25]:
df.isnull().sum().compute()

station_name        0
date                0
n_pixels            0
AOD_mean        28599
AOD_max         28599
AOD_p75         28599
AOD_count           0
AOD_coverage        0
dtype: int64

In [10]:
df1.head()

,station_name,date,PBLH_mean,PBLH_min,PBLH_max,TLML_mean,TLML_max,moisture_mean,moisture_max,SPEED_mean,SPEED_min,PRECTOT_sum,vent_coeff_mean,vent_coeff_min,inversion_proxy_mean,inversion_proxy_max,wind_dir_mean,PBLH_morning_mean,PBLH_morning_min
0,Sector 62 Noida,2023-01-01,397.121272,62.501790,1606.181754,287.391716,291.834930,4.210141,4.596994,1.996576,1.197900,1.446804e-17,747.056773,82.599158,3.282667,4.451987,285.534906,88.720278,62.501790
1,ITO Delhi,2023-01-01,399.623870,62.501815,1604.174628,287.399849,291.881780,4.193925,4.540281,1.971040,1.136127,1.328727e-17,762.165183,78.415429,3.267063,4.451969,278.414925,93.832402,62.501815
2,DTU Delhi,2023-01-01,402.661050,62.507588,1607.252416,287.387549,291.905809,4.274304,4.584352,2.060942,1.172816,1.679799e-17,803.143300,83.388538,3.244245,4.452061,279.795762,104.479195,62.507588
3,R K Puram Delhi,2023-01-01,398.314549,62.497915,1594.300259,287.406192,291.892512,4.132524,4.452865,1.874832,1.042732,9.165288e-18,747.714637,69.515044,3.268817,4.451898,270.369435,90.521994,62.497915
4,Punjabi Bagh Delhi,2023-01-01,403.378096,62.503915,1606.960470,287.406427,291.933358,4.206682,4.517335,1.990871,1.110166,1.403098e-17,788.303284,78.023516,3.247057,4.451984,273.979765,101.838645,62.503915


In [26]:
df1.isnull().sum().compute()

station_name            0
date                    0
PBLH_mean               0
PBLH_min                0
PBLH_max                0
TLML_mean               0
TLML_max                0
moisture_mean           0
moisture_max            0
SPEED_mean              0
SPEED_min               0
PRECTOT_sum             0
vent_coeff_mean         0
vent_coeff_min          0
inversion_proxy_mean    0
inversion_proxy_max     0
wind_dir_mean           0
PBLH_morning_mean       0
PBLH_morning_min        0
dtype: int64

In [28]:
print(df1.columns)
print(df1.shape[0].compute())

Index(['station_name', 'date', 'PBLH_mean', 'PBLH_min', 'PBLH_max',
       'TLML_mean', 'TLML_max', 'moisture_mean', 'moisture_max', 'SPEED_mean',
       'SPEED_min', 'PRECTOT_sum', 'vent_coeff_mean', 'vent_coeff_min',
       'inversion_proxy_mean', 'inversion_proxy_max', 'wind_dir_mean',
       'PBLH_morning_mean', 'PBLH_morning_min'],
      dtype='object')
71540


In [3]:
df_prefinal = dd.read_parquet(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_dataset.parquet")

In [7]:
df_final = dd.read_parquet(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\processed\final_ml_featured.parquet")

In [8]:
df_prefinal.head()

,date,station_name,pm25,latitude,longitude,n_pixels,AOD_mean,AOD_max,AOD_p75,AOD_count,...,SPEED_mean,SPEED_min,PRECTOT_sum,vent_coeff_mean,vent_coeff_min,inversion_proxy_mean,inversion_proxy_max,wind_dir_mean,PBLH_morning_mean,PBLH_morning_min
0,2023-01-01,Sector 62 Noida,131.027604,28.628,77.3649,5.0,0.978464,1.244282,0.978464,7.0,...,1.996576,1.197900,1.446804e-17,747.056773,82.599158,3.282667,4.451987,285.534906,88.720278,62.501790
1,2023-01-02,Sector 62 Noida,204.667292,28.628,77.3649,0.0,NaN,NaN,NaN,0.0,...,2.732568,1.221766,4.023247e-27,813.739627,133.028073,3.078406,4.451457,299.717118,102.379866,62.316191
2,2023-01-03,Sector 62 Noida,232.663958,28.628,77.3649,5.0,2.416098,2.570632,2.416098,3.0,...,5.108914,2.726804,0.000000e+00,3323.964466,219.997778,2.207823,4.450128,317.792988,90.863709,62.080795
3,2023-01-04,Sector 62 Noida,182.754687,28.628,77.3649,6.0,1.111868,1.203726,1.111868,4.0,...,4.043421,2.261521,0.000000e+00,1015.261216,216.160108,2.966831,4.451210,310.775510,404.106163,166.872214
4,2023-01-05,Sector 62 Noida,259.587500,28.628,77.3649,5.0,1.452860,1.886003,1.452860,4.0,...,3.574554,2.451186,0.000000e+00,589.089737,176.133804,3.404987,4.451395,323.640444,72.879655,61.412179


In [29]:
print(df_prefinal.columns)
print(df_prefinal.shape[0].compute())

Index(['date', 'station_name', 'pm25', 'latitude', 'longitude', 'n_pixels',
       'AOD_mean', 'AOD_max', 'AOD_p75', 'AOD_count', 'AOD_coverage',
       'PBLH_mean', 'PBLH_min', 'PBLH_max', 'TLML_mean', 'TLML_max',
       'moisture_mean', 'moisture_max', 'SPEED_mean', 'SPEED_min',
       'PRECTOT_sum', 'vent_coeff_mean', 'vent_coeff_min',
       'inversion_proxy_mean', 'inversion_proxy_max', 'wind_dir_mean',
       'PBLH_morning_mean', 'PBLH_morning_min'],
      dtype='object')
67970


In [30]:
# 1) Null values in each column
null_counts = df_prefinal.isnull().sum().compute().sort_values(ascending=False)
print("Null values per column:")
print(null_counts)

# Helper to find a column by common naming variants
def pick_col(columns, candidates):
    cols_lower = {c.lower(): c for c in columns}
    for c in candidates:
        if c.lower() in cols_lower:
            return cols_lower[c.lower()]
    return None

cols = list(df_prefinal.columns)

date_col = pick_col(cols, ["date", "datetime", "timestamp", "time", "day"])
lat_col = pick_col(cols, ["latitude", "lat"])
lon_col = pick_col(cols, ["longitude", "lon", "lng"])
station_col = pick_col(cols, ["station_name", "station", "stationid", "station_id", "site"])

# 2) Unique dates
if date_col is not None:
    unique_dates = (
        dd.to_datetime(df_prefinal[date_col], errors="coerce")
        .dropna()
        .dt.normalize()
        .drop_duplicates()
        .compute()
        .sort_values()
    )
    print(f"\nUnique dates ({len(unique_dates)}):")
    print(unique_dates)
else:
    print("\nNo date-like column found. Checked: date/datetime/timestamp/time/day")

# 3) Unique latitude-longitude pairs
if lat_col is not None and lon_col is not None:
    unique_lat_lon = (
        df_prefinal[[lat_col, lon_col]]
        .dropna()
        .drop_duplicates()
        .compute()
        .sort_values([lat_col, lon_col])
        .reset_index(drop=True)
    )
    print(f"\nUnique latitude-longitude pairs ({len(unique_lat_lon)}):")
    print(unique_lat_lon)
else:
    print("\nLatitude/Longitude columns not found. Checked: latitude/lat and longitude/lon/lng")

# 4) Unique station names
if station_col is not None:
    unique_stations = (
        df_prefinal[station_col]
        .dropna()
        .drop_duplicates()
        .compute()
        .sort_values()
        .reset_index(drop=True)
    )
    print(f"\nUnique station names ({len(unique_stations)}):")
    print(unique_stations)
else:
    print("\nNo station column found. Checked: station_name/station/stationid/station_id/site")

Null values per column:
AOD_mean                29921
AOD_p75                 29921
AOD_max                 29921
n_pixels                 2993
AOD_coverage             2993
AOD_count                2993
PBLH_mean                  92
moisture_mean              92
TLML_max                   92
TLML_mean                  92
PBLH_max                   92
PBLH_min                   92
PRECTOT_sum                92
vent_coeff_mean            92
vent_coeff_min             92
inversion_proxy_mean       92
inversion_proxy_max        92
moisture_max               92
SPEED_mean                 92
SPEED_min                  92
PBLH_morning_mean          92
wind_dir_mean              92
PBLH_morning_min           92
longitude                   0
date                        0
station_name                0
pm25                        0
latitude                    0
dtype: int64

Unique dates (731):
0       2023-01-01
1       2023-01-02
2       2023-01-03
3       2023-01-04
4       2023-01-05
       

In [9]:
df_final.head()


,date,station_name,pm25,latitude,longitude,n_pixels,AOD_mean,AOD_max,AOD_p75,AOD_count,...,AOD_lag1,AOD_lag2,AOD_roll3,AOD_roll7,station_pm_mean,station_pm_std,station_month_pm,proxy_anomaly,lat,lon
0,2023-01-03,AIIMS Raipur,54.811250,21.2575,81.5775,7,2.062045,2.549567,2.062045,7,...,-1.000000,-1.000000,2.062045,-1.000000,28.675352,18.036318,47.799848,0.079515,21.2575,81.5775
1,2023-01-04,AIIMS Raipur,65.033958,21.2575,81.5775,0,-1.000000,-1.000000,-1.000000,0,...,2.062045,-1.000000,2.062045,-1.000000,28.675352,18.036318,47.799848,-0.940005,21.2575,81.5775
2,2023-01-05,AIIMS Raipur,63.587917,21.2575,81.5775,7,2.145734,2.696037,2.145734,6,...,-1.000000,2.062045,2.103889,-1.000000,28.675352,18.036318,47.799848,0.071047,21.2575,81.5775
3,2023-01-06,AIIMS Raipur,56.248542,21.2575,81.5775,7,0.639584,0.949890,0.639584,7,...,2.145734,-1.000000,1.392659,1.615788,28.675352,18.036318,47.799848,0.070095,21.2575,81.5775
4,2023-01-07,AIIMS Raipur,51.333542,21.2575,81.5775,7,0.285947,0.349989,0.285947,7,...,0.639584,2.145734,1.023755,1.283327,28.675352,18.036318,47.799848,0.064576,21.2575,81.5775


In [31]:
df_final.isnull().sum().compute()

date                0
station_name        0
pm25                0
latitude            0
longitude           0
                   ..
station_pm_std      0
station_month_pm    0
proxy_anomaly       0
lat                 0
lon                 0
Length: 63, dtype: int64

In [11]:
dfpm25india = dd.read_parquet(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\insat_daily\insatdaily_grid.parquet")

In [13]:
dfpm25merra = dd.read_parquet(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\merra_daily\merra_daily_grid.parquet")

In [17]:
dfpm25india.head()



,lat_round,lon_round,date,AOD_mean,AOD_max,AOD_p75,AOD_count,AOD_coverage
0,7.55,77.35,2023-01-02,0.171448,0.208544,0.171448,2,0.285714
1,7.55,77.35,2023-01-03,0.191858,0.263068,0.191858,6,0.857143
2,7.55,77.35,2023-01-04,0.294082,0.314460,0.294082,4,0.571429
3,7.55,77.35,2023-01-05,0.233201,0.268513,0.233201,7,1.000000
4,7.55,77.35,2023-01-07,0.305637,0.387881,0.305637,7,1.000000


In [18]:
dfpm25merra.head()

,lat_round,lon_round,date,PBLH_mean,PBLH_min,PBLH_max,TLML_mean,TLML_max,moisture_mean,moisture_max,SPEED_mean,SPEED_min,PRECTOT_sum,vent_coeff_mean,vent_coeff_min,inversion_proxy_mean,inversion_proxy_max,wind_dir_mean,PBLH_morning_mean,PBLH_morning_min
0,7.5,77.5,2023-01-01,902.368225,693.783630,1093.887451,299.587128,300.549896,15.050393,15.502608,7.057846,5.248474,0.000001,6377.281250,4014.480469,0.337838,0.432336,218.531723,881.254883,859.190308
1,7.5,77.5,2023-01-02,1154.879028,1010.899536,1576.539795,299.099701,299.580170,14.156573,15.285531,8.155637,7.351015,0.000010,9462.201172,7581.365234,0.263144,0.295196,217.744308,1023.768738,1010.899536
2,7.5,77.5,2023-01-03,1233.849487,1011.184998,1668.796509,299.592010,299.903992,13.928544,14.930469,8.150998,7.276108,0.000025,9936.528320,8360.588867,0.251734,0.295884,218.241684,1586.278076,1573.884277
3,7.5,77.5,2023-01-04,1270.916626,1011.202942,1459.748169,299.886688,300.721802,13.815117,14.571502,9.271119,7.918664,0.000013,11807.703125,8885.229492,0.238691,0.295565,221.805923,1040.728271,1011.202942
4,7.5,77.5,2023-01-05,968.733337,688.263245,1455.347778,299.621246,300.677338,15.417347,16.923286,8.129073,6.266124,0.000140,7670.103027,5274.466309,0.328177,0.435424,229.257553,1262.005615,1143.311157


In [19]:
d1 = dd.read_parquet(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\insat_aod_merged.parquet")

In [20]:
d2 = dd.read_parquet(r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\interim\merra_merged.parquet")

In [21]:
d1.head()

,datetime,latitude,longitude,AOD
39351,2023-02-01 05:45:00,37.95,68.05,NaN
39352,2023-02-01 05:45:00,37.95,68.15,NaN
39353,2023-02-01 05:45:00,37.95,68.25,NaN
39354,2023-02-01 05:45:00,37.95,68.35,NaN
39355,2023-02-01 05:45:00,37.95,68.45,NaN


In [23]:
d1.isnull().sum().compute()

datetime             0
latitude             0
longitude            0
AOD          335204568
dtype: int64

In [22]:
d2.head()

,datetime,latitude,longitude,PBLH,TLML,QLML,ULML,VLML,SPEED,PRECTOT
0,2023-01-01 00:30:00,6.5,68.125,768.71521,299.251068,0.016384,-3.977004,-3.367228,5.214618,9.699725e-07
1,2023-01-01 00:30:00,6.5,68.750,752.21521,299.204193,0.016086,-3.867629,-3.223185,5.036883,6.440096e-07
2,2023-01-01 00:30:00,6.5,69.375,818.96521,299.297943,0.015598,-3.615676,-3.192423,4.825946,1.596636e-07
3,2023-01-01 00:30:00,6.5,70.000,1035.71521,299.594818,0.014808,-3.277786,-3.235392,4.607196,8.381903e-08
4,2023-01-01 00:30:00,6.5,70.625,1085.71521,299.516693,0.014816,-2.962356,-3.307169,4.443133,8.183997e-08


In [32]:
print(df_prefinal[df_prefinal['latitude'] > 38][['station_name','latitude','longitude']])

Dask DataFrame Structure:
              station_name latitude longitude
npartitions=1                                
                    string  float64    string
                       ...      ...       ...
Dask Name: getitem, 5 expressions
Expr=(Filter(frame=ReadParquetFSSpec(580f257), predicate=ReadParquetFSSpec(580f257)['latitude'] > 38))[['station_name', 'latitude', 'longitude']]


In [33]:
result = df_prefinal[df_prefinal['latitude'] > 38][
    ['station_name', 'latitude', 'longitude']
].compute()

print(result)

            station_name  latitude longitude
20903  Belur Math Howrah     82.63     88.36
20904  Belur Math Howrah     82.63     88.36
20905  Belur Math Howrah     82.63     88.36
20906  Belur Math Howrah     82.63     88.36
20907  Belur Math Howrah     82.63     88.36
...                  ...       ...       ...
55295  Belur Math Howrah     82.63     88.36
55296  Belur Math Howrah     82.63     88.36
55297  Belur Math Howrah     82.63     88.36
55298  Belur Math Howrah     82.63     88.36
55299  Belur Math Howrah     82.63     88.36

[713 rows x 3 columns]


In [35]:
import os

station_lat_lon = df_prefinal[["station_name", "latitude", "longitude"]].dropna().drop_duplicates().compute()
station_lat_lon = station_lat_lon.sort_values(["station_name", "latitude", "longitude"]).reset_index(drop=True)

output_dir = r"C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\raw\cpcb"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "station.csv")

station_lat_lon.to_csv(output_file, index=False)
print(f"Saved {len(station_lat_lon)} rows to {output_file}")

Saved 98 rows to C:\Users\kesha\OneDrive - dtu.ac.in\Desktop\aodtopm25\data\raw\cpcb\station.csv


In [6]:
df_prefinal.head()

,date,station_name,pm25,latitude,longitude,n_pixels,AOD_mean,AOD_max,AOD_p75,AOD_count,...,SPEED_mean,SPEED_min,PRECTOT_sum,vent_coeff_mean,vent_coeff_min,inversion_proxy_mean,inversion_proxy_max,wind_dir_mean,PBLH_morning_mean,PBLH_morning_min
0,2023-01-01,Sector 62 Noida,131.027604,28.628,77.3649,5.0,0.978464,1.244282,0.978464,7.0,...,1.996576,1.197900,1.446804e-17,747.056773,82.599158,3.282667,4.451987,285.534906,88.720278,62.501790
1,2023-01-02,Sector 62 Noida,204.667292,28.628,77.3649,0.0,NaN,NaN,NaN,0.0,...,2.732568,1.221766,4.023247e-27,813.739627,133.028073,3.078406,4.451457,299.717118,102.379866,62.316191
2,2023-01-03,Sector 62 Noida,232.663958,28.628,77.3649,5.0,2.416098,2.570632,2.416098,3.0,...,5.108914,2.726804,0.000000e+00,3323.964466,219.997778,2.207823,4.450128,317.792988,90.863709,62.080795
3,2023-01-04,Sector 62 Noida,182.754687,28.628,77.3649,6.0,1.111868,1.203726,1.111868,4.0,...,4.043421,2.261521,0.000000e+00,1015.261216,216.160108,2.966831,4.451210,310.775510,404.106163,166.872214
4,2023-01-05,Sector 62 Noida,259.587500,28.628,77.3649,5.0,1.452860,1.886003,1.452860,4.0,...,3.574554,2.451186,0.000000e+00,589.089737,176.133804,3.404987,4.451395,323.640444,72.879655,61.412179


In [7]:
df_prefinal.isnull().sum().compute()

date                        0
station_name                0
pm25                        0
latitude                    0
longitude                   0
n_pixels                 2993
AOD_mean                29921
AOD_max                 29921
AOD_p75                 29921
AOD_count                2993
AOD_coverage             2993
PBLH_mean                  92
PBLH_min                   92
PBLH_max                   92
TLML_mean                  92
TLML_max                   92
moisture_mean              92
moisture_max               92
SPEED_mean                 92
SPEED_min                  92
PRECTOT_sum                92
vent_coeff_mean            92
vent_coeff_min             92
inversion_proxy_mean       92
inversion_proxy_max        92
wind_dir_mean              92
PBLH_morning_mean          92
PBLH_morning_min           92
dtype: int64

In [8]:
df_prefinal = df_prefinal[df_prefinal["station_name"] != "Belur Math Howrah"]
